# Notebook 1 Outline: 01_part1_linreg_1feature 
## stellar luminosity 

## 1. setup

In [ ]:
# Install required libraries (run this once if needed)
%pip install numpy pandas matplotlib


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)


## 2. dataset

In [ ]:
M = [0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4]
L = [0.15, 0.35, 1.00, 2.30, 4.10, 7.00, 11.2, 17.5, 25.0, 35.0]

In [ ]:
plt.figure()
plt.scatter(M, L)
plt.xlabel("stellar mass")
plt.ylabel("stellar luminosity")
plt.title("Dataset: stellar mass M vs stellar luminosity L")
plt.show()

## 3. model
hypotesis function:
$$
f_{w,b}(M^{(i)})= L = w M^{(i)} + b
$$

In [ ]:
def predict(M, w, b):
    return w * M + b

w_test = 0.0
b_test = 0.0
L_hat_test = predict(M, w_test, b_test)
print("First 5 predictions with w=0, b=0:", L_hat_test[:5])

## 4. cost function
mean squeared error
$$
J(w,b) = \frac{1}{2n} \sum_{i=1}^{n} \big( f_{w,b}(M^{(i)}) - L^{(i)} \big)^2
$$

In [ ]:
def compute_cost(M, L, w, b):
    n = M.shape[0]
    L_hat = w * M + b  # f_{w,b}(M)
    errors = L_hat - L
    cost = (1 / (2 * n)) * np.sum(errors ** 2)
    return cost

print("Cost with w=0, b=0:", compute_cost(M, L, w_test, b_test))

## 4.1 cost function surface visualization

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

w_values = np.linspace(2.5, 3.5, 1000)
b_values = np.linspace(0.5, 1.5, 1000)

W, B = np.meshgrid(w_values, b_values)
J_vals = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        J_vals[i, j] = compute_cost(M, L, W[i, j], B[i, j])

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(W, B, J_vals, cmap=cm.viridis, linewidth=0, antialiased=True)
ax.set_xlabel("w")
ax.set_ylabel("b")
ax.set_zlabel("J(w,b)")
ax.set_title("Cost surface J(w,b)")
plt.show()

## 5. gradient descent
Analytical derivatives:
$$
\frac{\partial J}{\partial w} = \frac{1}{n} \sum_{i=1}^{n} \big( f_{w,b}(M^{(i)}) - L^{(i)} \big) M^{(i)}, \quad
\frac{\partial J}{\partial b} = \frac{1}{n} \sum_{i=1}^{n} \big( f_{w,b}(M^{(i)}) - L^{(i)} \big)
$$

In [ ]:
def compute_gradients(M, L, w, b):
    m = M.shape[0]
    L_hat = w * M + b  # f_{w,b}(M)
    errors = L_hat - L


    dj_dw = (1 / m) * np.sum(errors * M)
    dj_db = (1 / m) * np.sum(errors)
    return dj_dw, dj_db

dj_dw_test, dj_db_test = compute_gradients(M, L, w_test, b_test)
print("Gradients at w=0, b=0:", dj_dw_test, dj_db_test)

## 5.1 gradient descent implementation
Update rule:

Given a learning rate 
, we update:
$$
w := w - \alpha \frac{\partial J}{\partial w}, \quad
b := b - \alpha \frac{\partial J}{\partial b}
$$

In [ ]:
def gradient_descent(M_list, L_list, w_init, b_init, alpha, num_iterations):
    """Run gradient descent using explicit loops for gradients and cost."""
    w = w_init
    b = b_init
    history_iterations = []
    history_costs = []

    for i in range(num_iterations):
        dj_dw, dj_db = compute_gradients(M_list, L_list, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        cost = compute_cost(M_list, L_list, w, b)
        history_iterations.append(i)
        history_costs.append(cost)

        if i % max(1, (num_iterations // 10)) == 0:
            print(f"Iteration {i:4d}: w={w:7.4f}, b={b:7.4f}, cost={cost:8.4f}")

    return w, b, history_iterations, history_costs

alpha = 0.01
num_iterations = 1000

w_init = 1.0
b_init = 1.0

w_learned, b_learned, it_hist, cost_hist = gradient_descent(M, L, w_init, b_init, alpha, num_iterations)
print("\nLearned parameters:")
print("w =", w_learned)
print("b =", b_learned)

## Gradient descent (vectorized)

In [ ]:
def gradient_descent(M, L, w_init, b_init, alpha, num_iterations):
    w = w_init
    b = b_init
    history = []

    for i in range(num_iterations):
        dj_dw, dj_db = compute_gradients(M, L, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        cost = compute_cost(M, L, w, b)
        history.append((i, cost))

        if i % max(1, (num_iterations // 10)) == 0:
            print(f"Iteration {i:4d}: w={w:7.4f}, b={b:7.4f}, cost={cost:8.4f}")

    return w, b, history

alpha = 0.01
num_iterations = 2000

w_init = 0.0
b_init = 0.0

w_learned, b_learned, history = gradient_descent(M, L, w_init, b_init, alpha, num_iterations)
print("\nLearned parameters:")
print("w =", w_learned)
print("b =", b_learned)

## convergence

In [ ]:
iterations = [it for it, c in history]
costs = [c for it, c in history]

plt.figure()
plt.plot(iterations[15:], costs[15:])  # skip the first points
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.title("Gradient Descent: Cost vs Iterations")
plt.show()

The convergence starts rapidly, with a significant drop in cost during the initial iterations. As the algorithm gets closer to the minimum, the updates become smaller and the curve levels off, indicating slower but steady convergence. The consistent and gradual decrease in the loss suggests that the selected learning rate enables stable optimization without instability or divergence.

## experiments

In [ ]:
# Run gradient descent for multiple learning rates and report final w, b, and loss
learning_rates = [2e-3, 3e-3, 1e-2]
num_iters = 2000
w0, b0 = 0.0, 0.0

results = []
for alpha in learning_rates:
    w, b = w0, b0
    for _ in range(num_iters):
        dj_dw, dj_db = compute_gradients(M, L, w, b)
        w -= alpha * dj_dw
        b -= alpha * dj_db
    loss = compute_cost(M, L, w, b)
    results.append((alpha, w, b, loss))

for alpha, w, b, loss in results:
    print(f"alpha={alpha:.3g} -> w={w:.6f}, b={b:.6f}, loss={loss:.6f}")

## visualice the fitted line

In [ ]:
plt.figure()
plt.scatter(M, L, label="Data")
L_pred = predict(M, w_learned, b_learned)
plt.plot(M, L_pred, label="Fitted line")
plt.xlabel("M")
plt.ylabel("L")
plt.title("Linear Regression Fit (one feature)")
plt.legend()
plt.show()